In [ ]:
from sklearn.datasets import fetch_california_housing

In [ ]:
def SGD_Qlearning(X, y, beta0, initial_eta=0.001, n_b=1, n_ep=100, tol=3e-1, reg_lambda=0.01):
    n, p = X.shape
    beta_t = beta0
    eta = initial_eta
    beta_lst, fx_lst = [], []

    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X = (X - X_mean) / (X_std + 1e-8)

    actions = [0.9, 1, 1.1]
    states = ['improved', 'same', 'worse']
    Q_table = np.zeros((len(states), len(actions)))

    previous_loss = float('inf')

    for t in range(n_ep):
        predictions = X.dot(beta_t)
        residuals = predictions - y
        print(la.norm(residuals))
        f_val = (0.5/n) * (la.norm(residuals)**2) + (reg_lambda/2) * (la.norm(beta_t)**2)
        beta_lst.append(beta_t)
        fx_lst.append(f_val)

        if f_val < previous_loss - tol:
            state = 0
        elif abs(f_val - previous_loss) <= tol:
            state = 1
        else:
            state = 2

        action = np.argmax(Q_table[state])
        eta *= actions[action]

        for it in range(int(n/n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X[batch, :]
            y_b = y[batch]
            predictions_b = X_b.dot(beta_t)
            residuals_b = predictions_b - y_b
            df_t = (1/n_b) * (X_b.T.dot(residuals_b)) + reg_lambda * beta_t
            beta_t -= eta * df_t

        reward = previous_loss - f_val
        Q_table[state, action] = 0.9 * Q_table[state, action] + 0.1 * reward

        previous_loss = f_val

        if la.norm(df_t) <= tol:
            print("Convergence achieved!")
            break

        print("Epoch: {:4d}, Loss = {:.3e}, Learning Rate = {:.3e}".format(t, f_val, eta))

    return beta_lst, fx_lst

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def evaluate_accuracy(X, y, weights):
    predictions = np.argmax(X.dot(weights), axis=1)
    true_labels = np.argmax(y, axis=1)
    return accuracy_score(true_labels, predictions)

def train_SGD_fixed(X_train, y_train, X_test, y_test, beta0, eta=0.01, n_b=1, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0
    beta_lst, fx_lst, accuracy_lst = [], [], []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X_train[batch, :]
            y_b = y_train[batch]
            residuals = X_b.dot(beta_t) - y_b
            grad = (1 / n_b) * X_b.T.dot(residuals)
            beta_t -= eta * grad

        f_val = (0.5 / n) * np.linalg.norm(X_train.dot(beta_t) - y_train) ** 2
        beta_lst.append(beta_t.copy())
        fx_lst.append(f_val)
        accuracy = evaluate_accuracy(X_test, y_test, beta_t)
        accuracy_lst.append(accuracy)
        print(f"Epoch: {t}, Loss = {f_val:.3e}, Accuracy = {accuracy:.4f}")

    return beta_lst, fx_lst, accuracy_lst

def train_SGD_Qlearning(X_train, y_train, X_test, y_test, beta0, initial_eta=0.01, n_b=1, n_ep=100, tol=3e-1, reg_lambda=0.01):
    n, p = X_train.shape
    beta_t = beta0
    eta = initial_eta
    beta_lst, fx_lst, accuracy_lst = [], [], []

    actions = [0.9, 1, 1.1]
    states = ['improved', 'same', 'worse']
    Q_table = np.zeros((len(states), len(actions)))

    previous_loss = float('inf')

    for t in range(n_ep):
        predictions = X_train.dot(beta_t)
        residuals = predictions - y_train
        f_val = (0.5 / n) * (np.linalg.norm(residuals) ** 2) + (reg_lambda / 2) * (np.linalg.norm(beta_t) ** 2)
        beta_lst.append(beta_t.copy())
        fx_lst.append(f_val)

        accuracy = evaluate_accuracy(X_test, y_test, beta_t)
        accuracy_lst.append(accuracy)

        print(f"Epoch: {t}, Loss = {f_val:.3e}, Learning Rate = {eta:.3e}, Accuracy = {accuracy:.4f}")

        if f_val < previous_loss - tol:
            state = 0
        elif abs(f_val - previous_loss) <= tol:
            state = 1
        else:
            state = 2

        action = np.argmax(Q_table[state])
        eta *= actions[action]

        for _ in range(int(n / n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X_train[batch, :]
            y_b = y_train[batch]
            residuals_b = X_b.dot(beta_t) - y_b
            grad_b = (1 / n_b) * (X_b.T.dot(residuals_b)) + reg_lambda * beta_t
            beta_t -= eta * grad_b

        reward = previous_loss - f_val
        Q_table[state, action] = 0.9 * Q_table[state, action] + 0.1 * reward
        previous_loss = f_val

    return beta_lst, fx_lst, accuracy_lst

def train_AdaGrad(X_train, y_train, X_test, y_test, beta0, eta=0.01, n_b=1, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0
    grad_squared = np.zeros_like(beta_t)
    epsilon = 1e-8

    beta_lst, fx_lst, accuracy_lst = [], [], []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X_train[batch, :]
            y_b = y_train[batch]
            residuals = X_b.dot(beta_t) - y_b
            grad = (1 / n_b) * X_b.T.dot(residuals)
            grad_squared += grad**2

            beta_t -= eta * grad / (np.sqrt(grad_squared) + epsilon)

        f_val = (0.5 / n) * np.linalg.norm(X_train.dot(beta_t) - y_train) ** 2
        beta_lst.append(beta_t.copy())
        fx_lst.append(f_val)
        accuracy = evaluate_accuracy(X_test, y_test, beta_t)
        accuracy_lst.append(accuracy)
        print(f"Epoch: {t}, Loss = {f_val:.3e}, Accuracy = {accuracy:.4f}")

    return beta_lst, fx_lst, accuracy_lst

def train_Adam(X_train, y_train, X_test, y_test, beta0, eta=0.001, n_b=1, n_ep=100, beta1=0.9, beta2=0.999):
    n, p = X_train.shape
    beta_t = beta0
    mt, vt = np.zeros_like(beta_t), np.zeros_like(beta_t)
    epsilon = 1e-8

    beta_lst, fx_lst, accuracy_lst = [], [], []

    for t in range(1, n_ep + 1):
        for _ in range(int(n / n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X_train[batch, :]
            y_b = y_train[batch]
            residuals = X_b.dot(beta_t) - y_b
            grad = (1 / n_b) * X_b.T.dot(residuals)

            mt = beta1 * mt + (1 - beta1) * grad
            vt = beta2 * vt + (1 - beta2) * grad**2

            mt_hat = mt / (1 - beta1**t)
            vt_hat = vt / (1 - beta2**t)

            beta_t -= eta * mt_hat / (np.sqrt(vt_hat) + epsilon)

        f_val = (0.5 / n) * np.linalg.norm(X_train.dot(beta_t) - y_train) ** 2
        beta_lst.append(beta_t.copy())
        fx_lst.append(f_val)
        accuracy = evaluate_accuracy(X_test, y_test, beta_t)
        accuracy_lst.append(accuracy)
        print(f"Epoch: {t}, Loss = {f_val:.3e}, Accuracy = {accuracy:.4f}")

    return beta_lst, fx_lst, accuracy_lst

def train_SGD_Nesterov(X_train, y_train, X_test, y_test, beta0, eta=0.01, momentum=0.9, n_b=1, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0
    velocity = np.zeros_like(beta_t)

    beta_lst, fx_lst, accuracy_lst = [], [], []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            batch = np.random.randint(0, n, n_b)
            X_b = X_train[batch, :]
            y_b = y_train[batch]
            lookahead = beta_t + momentum * velocity
            residuals = X_b.dot(lookahead) - y_b
            grad = (1 / n_b) * X_b.T.dot(residuals)

            velocity = momentum * velocity - eta * grad
            beta_t += velocity

        f_val = (0.5 / n) * np.linalg.norm(X_train.dot(beta_t) - y_train) ** 2
        beta_lst.append(beta_t.copy())
        fx_lst.append(f_val)
        accuracy = evaluate_accuracy(X_test, y_test, beta_t)
        accuracy_lst.append(accuracy)
        print(f"Epoch: {t}, Loss = {f_val:.3e}, Accuracy = {accuracy:.4f}")

    return beta_lst, fx_lst, accuracy_lst

mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

combined_images = np.concatenate((train_images, test_images), axis=0)
combined_labels = np.concatenate((train_labels, test_labels), axis=0)

selected_images = combined_images[:10000]
selected_labels = combined_labels[:10000]

train_images, test_images, train_labels, test_labels = train_test_split(
    selected_images, selected_labels, test_size=0.2, random_state=42)

train_images = train_images.reshape((8000, 28 * 28)).astype('float32') / 255
test_images = test_images.reshape((2000, 28 * 28)).astype('float32') / 255

train_labels = tf.keras.utils.to_categorical(train_labels, 10)
test_labels = tf.keras.utils.to_categorical(test_labels, 10)

beta0 = np.random.randn(28 * 28, 10)

learning_rates = [0.01, 0.001, 0.0001]
accuracies = {}

for eta in learning_rates:
    print(f"\nTraining with fixed learning rate = {eta}")
    _, _, acc_lst = train_SGD_fixed(train_images, train_labels, test_images, test_labels, beta0= np.random.randn(28 * 28, 10), eta=eta)
    accuracies[f'Fixed LR {eta}'] = acc_lst

_, _, acc_lst_Qlearning = train_SGD_Qlearning(train_images, train_labels, test_images, test_labels, beta0= np.random.randn(28 * 28, 10))
_, _, acc_lst_AdaGrad = train_AdaGrad(train_images, train_labels, test_images, test_labels, beta0= np.random.randn(28 * 28, 10))
_, _, acc_lst_Adam = train_Adam(train_images, train_labels, test_images, test_labels, beta0= np.random.randn(28 * 28, 10))
_, _, acc_lst_SGDNesterov = train_SGD_Nesterov(train_images, train_labels, test_images, test_labels, beta0= np.random.randn(28 * 28, 10))

accuracies['Q-learning'] = acc_lst_Qlearning
accuracies['AdaGrad'] = acc_lst_AdaGrad
accuracies['Adam'] = acc_lst_Adam
accuracies['SGD Nesterov'] = acc_lst_SGDNesterov

plt.figure(figsize=(12, 8))
for key, acc_lst in accuracies.items():
    plt.plot(acc_lst, label=f'{key}')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('Accuracy over Epochs for Different Learning Rate Strategies')
plt.legend()
plt.show()


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Training with fixed learning rate = 0.01
Epoch: 0, Loss = 7.476e+00, Accuracy = 0.2455
Epoch: 1, Loss = 2.843e+00, Accuracy = 0.2760
Epoch: 2, Loss = 3.679e+00, Accuracy = 0.2965
Epoch: 3, Loss = 1.273e+00, Accuracy = 0.4940
Epoch: 4, Loss = 1.145e+00, Accuracy = 0.4465
Epoch: 5, Loss = 8.214e-01, Accuracy = 0.5680
Epoch: 6, Loss = 7.118e-01, Accuracy = 0.6235
Epoch: 7, Loss = 6.725e-01, Accuracy = 0.5985
Epoch: 8, Loss = 1.236e+00, Accuracy = 0.4540
Epoch: 9, Loss = 7.507e-01, Accuracy = 0.5590
Epoch: 10, Loss = 6.474e-01, Accuracy = 0.5670
Epoch: 11, Loss = 6.324e-01, Accuracy = 0.5750
Epoch: 12, Loss = 4.849e-01, Accuracy = 0.6515
Epoch: 13, Loss = 4.819e-01, Accuracy = 0.6650
Epoch: 14, Loss = 6.069e-01, Accuracy = 0.5090
Epoch: 15, Loss = 5.292e-01, Accuracy = 0.6125
Epoch: 16, Loss = 5.011e-01, Accuracy = 0.6805
Epoch: 17, Loss = 5.557e-01, Accuracy = 0.5725
Epoch: 18, Loss = 5.618e-01, Accuracy = 0.5470
Epoch: 19, Loss = 4.288e

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

def evaluate_mse(X, y, weights):
    predictions = X.dot(weights)
    mse = mean_squared_error(y, predictions)
    return mse

def train_SGD_fixed(X_train, y_train, X_test, y_test, beta0, eta=0.01, n_b=32, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0.copy()
    mse_lst = []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            idx = np.random.randint(0, n, n_b)
            X_b, y_b = X_train[idx], y_train[idx]
            grad = (1 / n_b) * X_b.T.dot(X_b.dot(beta_t) - y_b)
            beta_t -= eta * grad

        mse = evaluate_mse(X_test, y_test, beta_t)
        mse_lst.append(mse)
        print(f"Epoch: {t}, Test MSE = {mse:.4f}")

    return mse_lst

def train_SGD_Qlearning(X_train, y_train, X_test, y_test, beta0, initial_eta=0.0005, n_b=32, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0.copy()
    eta = initial_eta
    mse_lst = []

    actions = [0.9, 1.0, 1.1]
    states = ['improved', 'same', 'worse']
    Q_table = np.zeros((len(states), len(actions)))

    prev_loss = float('inf')

    for t in range(n_ep):
        preds = X_train.dot(beta_t)
        current_loss = np.mean((preds - y_train) ** 2)

        if current_loss < prev_loss - 1e-3:
            state = 0
        elif abs(current_loss - prev_loss) <= 1e-3:
            state = 1
        else:
            state = 2

        action = np.argmax(Q_table[state])
        eta *= actions[action]

        for _ in range(int(n / n_b)):
            idx = np.random.randint(0, n, n_b)
            X_b, y_b = X_train[idx], y_train[idx]
            grad = (1 / n_b) * X_b.T.dot(X_b.dot(beta_t) - y_b)
            beta_t -= eta * grad
        mse = evaluate_mse(X_test, y_test, beta_t)
        mse_lst.append(mse)
        reward = prev_loss - current_loss
        Q_table[state, action] = 0.9 * Q_table[state, action] + 0.1 * reward
        prev_loss = current_loss
        print(f"Epoch: {t}, LR = {eta:.6f}, Test MSE = {mse:.4f}")

    return mse_lst

def train_AdaGrad(X_train, y_train, X_test, y_test, beta0, eta=0.01, n_b=32, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0.copy()
    grad_squared = np.zeros_like(beta_t)
    epsilon = 1e-8
    mse_lst = []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            idx = np.random.randint(0, n, n_b)
            X_b, y_b = X_train[idx], y_train[idx]
            grad = (1 / n_b) * X_b.T.dot(X_b.dot(beta_t) - y_b)
            grad_squared += grad ** 2
            adj_grad = grad / (np.sqrt(grad_squared) + epsilon)
            beta_t -= eta * adj_grad

        mse = evaluate_mse(X_test, y_test, beta_t)
        mse_lst.append(mse)
        print(f"Epoch: {t}, Test MSE = {mse:.4f}")

    return mse_lst

def train_Adam(X_train, y_train, X_test, y_test, beta0, eta=0.001, n_b=32, n_ep=100, beta1=0.9, beta2=0.999):
    n, p = X_train.shape
    beta_t = beta0.copy()
    m = np.zeros_like(beta_t)
    v = np.zeros_like(beta_t)
    epsilon = 1e-8
    mse_lst = []

    for t in range(1, n_ep + 1):
        for _ in range(int(n / n_b)):
            idx = np.random.randint(0, n, n_b)
            X_b, y_b = X_train[idx], y_train[idx]
            grad = (1 / n_b) * X_b.T.dot(X_b.dot(beta_t) - y_b)
            m = beta1 * m + (1 - beta1) * grad
            v = beta2 * v + (1 - beta2) * (grad ** 2)
            m_hat = m / (1 - beta1 ** t)
            v_hat = v / (1 - beta2 ** t)
            beta_t -= eta * m_hat / (np.sqrt(v_hat) + epsilon)

        mse = evaluate_mse(X_test, y_test, beta_t)
        mse_lst.append(mse)
        print(f"Epoch: {t}, Test MSE = {mse:.4f}")

    return mse_lst

def train_SGD_Nesterov(X_train, y_train, X_test, y_test, beta0, eta=0.001, momentum=0.9, n_b=32, n_ep=100):
    n, p = X_train.shape
    beta_t = beta0.copy()
    velocity = np.zeros_like(beta_t)
    mse_lst = []

    for t in range(n_ep):
        for _ in range(int(n / n_b)):
            idx = np.random.randint(0, n, n_b)
            X_b, y_b = X_train[idx], y_train[idx]
            lookahead = beta_t + momentum * velocity
            grad = (1 / n_b) * X_b.T.dot(X_b.dot(lookahead) - y_b)
            velocity = momentum * velocity - eta * grad
            beta_t += velocity

        mse = evaluate_mse(X_test, y_test, beta_t)
        mse_lst.append(mse)
        print(f"Epoch: {t}, Test MSE = {mse:.4f}")

    return mse_lst

np.random.seed(0)
beta0 = np.random.randn(X_train_scaled.shape[1])
results = {}

print("\nSGD Fixed LR")
results['SGD Fixed'] = train_SGD_fixed(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled, beta0.copy(), eta=0.00009)

print("\nSGD Q-learning")
results['SGD Q-learning'] = train_SGD_Qlearning(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled, beta0.copy(), initial_eta=0.0005)

print("\nAdaGrad")
results['AdaGrad'] = train_AdaGrad(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled, beta0.copy(), eta=0.01)

print("\nAdam")
results['Adam'] = train_Adam(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled, beta0.copy(), eta=0.001)

print("\nSGD Nesterov")
results['SGD Nesterov'] = train_SGD_Nesterov(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled, beta0.copy(), eta=0.001)

plt.figure(figsize=(12, 8))
for key, mse_lst in results.items():
    plt.plot(mse_lst, label=key)

plt.xlabel('Epoch')
plt.ylabel('Test MSE')
plt.title('Test MSE over Epochs for Different Optimizers (California Housing)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
